# Wan 2.2 Kaggle: independent scientific shots

Fresh controlled notebook for Wan 2.2 I2V-A14B on two Tesla T4 GPUs. It tests independent reference-image shots first, then an optional reduced-area comparison and optional chaining comparison.


In [ ]:
import os, subprocess
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = Path("/kaggle/working")
IMAGE = Path("/kaggle/input/datasets/abhinavk0006/testimage/1.jpeg")
REPOS = {
    "Wan2.2": ("https://github.com/abhinavk0006/Wan2.2.git", "78fb5ce"),
    "image-to-video-pipeline": ("https://github.com/abhinavk0006/image-to-video-pipeline.git", "65290bb"),
    "Edu-video-gen-dataset": ("https://github.com/abhinavk0006/Edu-video-gen-dataset.git", "02b327b"),
}
for name, (url, commit) in REPOS.items():
    path = ROOT / name
    if not path.exists(): subprocess.run(["git", "clone", url, str(path)], check=True)
    subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(path), "checkout", "--detach", commit], check=True)
assert IMAGE.is_file(), f"Upload the reference image and update IMAGE: {IMAGE}"
print("Pinned repositories:", REPOS)


In [ ]:
%pip install -q -r /kaggle/working/Edu-video-gen-dataset/requirements.txt imageio-ffmpeg
%pip install -q 'diffusers>=0.31,<0.33' 'transformers>=4.49,<=4.51.3' 'accelerate>=1.1.1' easydict safetensors ftfy huggingface_hub imageio


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 2), "GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2


## Create a two-shot fixture

Both clips use the same verified image independently. The durations differ to validate per-shot duration handling.


In [ ]:
import json
pipeline = ROOT / "image-to-video-pipeline"
fixture_path = pipeline / "kaggle_experiments" / "independent_test_tube_two_shots.json"
fixture_path.parent.mkdir(exist_ok=True)
fixture = {
  "name": "Independent Test Tube Shots",
  "description": "Two independent shots from one verified reference.",
  "subject": "Laboratory demonstration",
  "educational_goal": "Show a controlled droplet entering a test tube.",
  "knowledge_source": "manual_json",
  "clips": [
    {"name": "setup_motion", "scene": {"scene": "Macro test tube held by a clamp with a pipette above it.", "camera": "Locked macro shot", "lighting": "Soft neutral laboratory lighting"}, "prompt_bundle": {"motion_prompt": "A pipette slowly releases one clear droplet into the test tube. Keep the camera, tube, clamp, pipette, and white crystals stable.", "negative_prompt": "camera shake, zoom, cuts, extra tubes, text, labels, distorted glass", "clip_duration_seconds": 2.0}},
    {"name": "settled_followup", "scene": {"scene": "The same macro test-tube setup after the droplet has entered.", "camera": "Locked macro shot", "lighting": "Soft neutral laboratory lighting"}, "prompt_bundle": {"motion_prompt": "Hold the same test-tube composition steady. Show subtle settling after the droplet enters; do not change the crystals or apparatus.", "negative_prompt": "camera shake, zoom, cuts, extra tubes, text, labels, changing substance, distorted glass", "clip_duration_seconds": 1.0}}
  ]
}
fixture_path.write_text(json.dumps(fixture, indent=2), encoding="utf-8")
print(fixture_path)


## Native-area baseline

Run this first. It uses independent shots, Lightning 4-step inference, 5/11 GiB placement, and telemetry.


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/independent_test_tube_two_shots.json --initial-image /kaggle/input/datasets/abhinavk0006/testimage/1.jpeg --output-dir /kaggle/working/wan_independent_native --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode independent --generator-arg=--lightning --generator-arg=--gpu0-memory-gib=5 --generator-arg=--gpu1-memory-gib=11 --generator-arg=--memory-telemetry


In [ ]:
from pathlib import Path
for path in sorted(Path("/kaggle/working/wan_independent_native").rglob("*.mp4")):
    print(path, path.stat().st_size, "bytes")


## Reduced-area comparison

Only run after the native baseline. This keeps prompts, frames, and placement fixed while reducing activation pressure.


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/independent_test_tube_two_shots.json --initial-image /kaggle/input/datasets/abhinavk0006/testimage/1.jpeg --output-dir /kaggle/working/wan_independent_low_area --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode independent --wan-max-area 345600 --generator-arg=--lightning --generator-arg=--gpu0-memory-gib=5 --generator-arg=--gpu1-memory-gib=11 --generator-arg=--memory-telemetry


## Optional chained comparison

Run only after the independent baseline succeeds. This measures the visual-drift trade-off by feeding the previous generated clip frame into the next shot.


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/independent_test_tube_two_shots.json --initial-image /kaggle/input/datasets/abhinavk0006/testimage/1.jpeg --output-dir /kaggle/working/wan_chained_native --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode chain --generator-arg=--lightning --generator-arg=--gpu0-memory-gib=5 --generator-arg=--gpu1-memory-gib=11 --generator-arg=--memory-telemetry
